In [3]:
pip install duckdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 30.8 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [17]:
import duckdb
import pandas as pd

In [19]:
# Pass latin1 encoding to handle non-UTF-8 characters
df = pd.read_csv('datacosupplychaindataset.csv', encoding='latin1')

print("Data loaded successfully! Shape:", df.shape)
df.head()

Data loaded successfully! Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [ ]:
con = duckdb.connect('supply_chain.duckdb')
con.register('df_view', df)

In [ ]:
print(df.columns.tolist())

order date (DateOrders)

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrde

In [ ]:
# Clean string columns natively in Pandas to avoid SQL TRIM binder errors
string_cols = [
    'Delivery Status', 'Late_delivery_risk', 'Shipping Mode', 
    'Order Status', 'Category Name', 'Customer City', 
    'Customer Country', 'Order City', 'Order Country', 
    'Order Region', 'Market'
]

for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Handle missing geographic values
df['Order City'] = df['Order City'].replace(['nan', 'None', ''], 'Unknown City')
df['Order Country'] = df['Order Country'].replace(['nan', 'None', ''], 'Unknown Country')

# Parse US date formats safely
df['order_date_parsed'] = pd.to_datetime(df['order date (DateOrders)'], format='%m/%d/%Y %H:%M', errors='coerce')
df['shipping_date_parsed'] = pd.to_datetime(df['shipping date (DateOrders)'], format='%m/%d/%Y %H:%M', errors='coerce')

In [ ]:

# Connect to DuckDB and register DataFrame
con.register('df_clean_view', df)

# 6. Ingest into DuckDB raw table
con.execute("""
CREATE OR REPLACE TABLE raw_supply_chain AS 
SELECT 
    CAST("Order Id" AS BIGINT) AS order_id,
    CAST("Order Item Id" AS BIGINT) AS order_item_id,
    order_date_parsed AS order_date,
    shipping_date_parsed AS shipping_date,
    CAST("Days for shipping (real)" AS INTEGER) AS actual_shipping_days,
    CAST("Days for shipment (scheduled)" AS INTEGER) AS scheduled_shipping_days,
    "Delivery Status" AS delivery_status,
    "Late_delivery_risk" AS late_delivery_risk,
    "Shipping Mode" AS shipping_mode,
    "Order Status" AS order_status,
    "Category Name" AS category_name,
    "Customer City" AS customer_city,
    "Customer Country" AS customer_country,
    "Order City" AS order_city,
    "Order Country" AS order_country,
    "Order Region" AS order_region,
    "Market" AS market,
    CAST("Order Item Quantity" AS INTEGER) AS item_quantity,
    CAST("Order Item Profit Ratio" AS DECIMAL(10,4)) AS profit_ratio,
    CAST("Sales" AS DECIMAL(10,2)) AS sales_amount,
    CAST("Order Item Total" AS DECIMAL(10,2)) AS order_item_total
FROM df_clean_view;
""")

print("Successfully loaded rows into raw_supply_chain:", con.execute("SELECT COUNT(*) FROM raw_supply_chain").fetchone()[0])

Successfully loaded rows into raw_supply_chain: 180519


In [35]:
con.execute("""
CREATE OR REPLACE VIEW stg_supply_chain AS
SELECT 
    order_id,
    order_item_id,
    order_date,
    shipping_date,
    actual_shipping_days,
    scheduled_shipping_days,
    
    -- Lead time variance in days
    (actual_shipping_days - scheduled_shipping_days) AS lead_time_variance_days,
    
    -- SLA Status Classification
    CASE 
        WHEN actual_shipping_days > scheduled_shipping_days THEN 'Late Delivery'
        WHEN actual_shipping_days < scheduled_shipping_days THEN 'Advance Shipping'
        ELSE 'On-Time Shipping'
    END AS calculated_delivery_status,
    
    delivery_status AS raw_delivery_status,
    shipping_mode,
    order_status,
    order_city,
    order_country,
    order_region,
    market,
    item_quantity,
    sales_amount,
    order_item_total,
    profit_ratio
FROM raw_supply_chain
WHERE order_status NOT IN ('SUSPECTED_FRAUD', 'CANCELED')
  AND actual_shipping_days >= 0
  AND sales_amount > 0;
""")

print("Staging view created successfully!")

Staging view created successfully!


In [36]:
# 1. Intermediate Order Fulfillment Summary
con.execute("""
CREATE OR REPLACE TABLE int_order_fulfillment_summary AS
SELECT 
    order_id,
    MIN(order_date) AS order_date,
    MIN(shipping_date) AS shipping_date,
    order_region,
    market,
    shipping_mode,
    COUNT(DISTINCT order_item_id) AS total_line_items,
    SUM(item_quantity) AS total_item_units,
    SUM(order_item_total) AS total_order_value,
    MAX(actual_shipping_days) AS actual_shipping_days,
    MAX(scheduled_shipping_days) AS scheduled_shipping_days,
    MAX(actual_shipping_days) - MAX(scheduled_shipping_days) AS max_lead_time_variance,
    CASE WHEN MAX(actual_shipping_days) > MAX(scheduled_shipping_days) THEN 1 ELSE 0 END AS is_late_delivery
FROM stg_supply_chain
GROUP BY order_id, order_region, market, shipping_mode;
""")

# 2. Fact Table: Carrier & Regional SLA Performance
con.execute("""
CREATE OR REPLACE TABLE fct_carrier_sla_performance AS
SELECT 
    order_region,
    shipping_mode,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(is_late_delivery) AS total_late_orders,
    ROUND(CAST(SUM(is_late_delivery) AS DOUBLE) / COUNT(DISTINCT order_id) * 100, 2) AS late_delivery_rate_pct,
    ROUND(AVG(actual_shipping_days), 2) AS avg_actual_shipping_days,
    ROUND(AVG(scheduled_shipping_days), 2) AS avg_scheduled_shipping_days,
    ROUND(AVG(max_lead_time_variance), 2) AS avg_delay_variance_days
FROM int_order_fulfillment_summary
GROUP BY order_region, shipping_mode
HAVING COUNT(DISTINCT order_id) >= 100
ORDER BY late_delivery_rate_pct DESC;
""")

# Inspect the top 5 worst routes by SLA breach rate
con.execute("SELECT * FROM fct_carrier_sla_performance LIMIT 5").df()

,order_region,shipping_mode,total_orders,total_late_orders,late_delivery_rate_pct,avg_actual_shipping_days,avg_scheduled_shipping_days,avg_delay_variance_days
0,South Asia,First Class,523,523.0,100.0,2.0,1.0,1.0
1,Oceania,First Class,566,566.0,100.0,2.0,1.0,1.0
2,West Asia,First Class,316,316.0,100.0,2.0,1.0,1.0
3,Southern Europe,First Class,507,507.0,100.0,2.0,1.0,1.0
4,Eastern Asia,First Class,469,469.0,100.0,2.0,1.0,1.0
